<a href="https://colab.research.google.com/github/hsschachter/docs/blob/gh-pages/HW1_Word_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 159/259
#<center> Homework 1: Word Embeddings </center>

<center> Due: February 4, 2025 @ 11:59pm </center>

### Introduction

**MAKE A COPY OF THIS NOTEBOOK AND WORK ON THAT. If you do not work on a copy of this, your work will be erased.**

In this homework, you will be working with word embeddings.

In class, we've discussed [words](https://web.stanford.edu/~jurafsky/slp3/2.pdf) as dimensionality reduction, as well as the implications in choosing appropriate measures of **types** and **tokens**. In discussing words as singular units, and about the act of **tokenizing**, we've developed understanding of why the choice of delimiters can change the units and resulting findings.  We've discussed how the representation of words is informed by the *distributional hypothesis* (we understand the meaning of a word by the distribution of contexts in which it is used).

As explored in [Jurafsky and Martin](https://web.stanford.edu/~jurafsky/slp3/6.pdf), word embeddings are learned vector representations of words. In this homework, we are going to work with dense word embedding models.

If you have never used Google Colab before, a [cheat sheet](https://colab.research.google.com/github/Tanu-N-Prabhu/Python/blob/master/Cheat_sheet_for_Google_Colab.ipynb#scrollTo=jwK2yGZZDeLH) may be found here.




> In this homework, a CPU runtime will work, so don't waste any GPU resources you might have (which you'll want later this semester). Be sure the CPU runtime is active by going to: **Runtime > Change runtime rype > CPU**.



### Q0: Nearest Neighbors with Word Embeddings

Once we have a learned representation of a text we are working with, we can use our model to understand the meaning of a word based on its neighbours. Semantically similar words have vector representations or **embeddings** that are similar, as described in [6.8](https://web.stanford.edu/~jurafsky/slp3/6.pdf). Let's look at this with a pre-trained model.



In [ ]:
import numpy as np
from gensim.models import KeyedVectors
import operator
from sklearn.decomposition import PCA

#### Set-up, Introducing GloVe

We are going to work with a pretrained [GloVe word embedding model](https://nlp.stanford.edu/projects/glove/) that has 100,000 words in its vocabulary and 100 dimensions.

In [ ]:
!wget https://github.com/dbamman/nlp25/raw/main/data/glove.6B.100d.100K.w2v.txt

In [ ]:
glove_100=KeyedVectors.load_word2vec_format("glove.6B.100d.100K.w2v.txt", binary=False)
print("Vocab size: ", len(glove_100.index_to_key))
print("Dimension: ", glove_100.vector_size)

Each word is represented as a 100-dimensional vector in our model. We can access the vector for a given word by passing the word as a **key** to our KeyedVector word embedding model, which is simply a structure for accessing learned embeddings. This will return an array which we can process.

In [ ]:
glove_100['burger']

#### A. Cosine Similarity


We are now going to write a function to retreive the top-k similar words to a given word from our pre-trained word embedding model. We are going to use the **cosine similarity** of the vector representation of words to calculate the semantic similarity between words.

$$
cosine\_similarity(A,B)= (A \cdot B) \div (||A|| \times ||B||)
$$

In [ ]:
def cosine_similarity(vector1, vector2):
  dot_product=np.dot(vector1, vector2)
  maginitude_vector1=np.linalg.norm(vector1)
  maginitude_vector2=np.linalg.norm(vector2)
  cosine= dot_product / (maginitude_vector1 * maginitude_vector2)
  return cosine

In [ ]:
cosine_similarity(glove_100['burger'], glove_100['fries'])

In [ ]:
cosine_similarity(glove_100['burger'], glove_100['pen'])

As we would expect, the cosine similarity between the words 'burger' and 'fries' is higher than the cosine similarity between the words 'burger' and 'pen'.


#### B. Top-k nearest neighbors


We will now use this function to score the similarity between our target word and all other words in our vocabulary so we can find the top-k most similar words.

In [ ]:
def find_k_nearest_neighbors(model, query, k):
  similarity_scores={}
  for word in model.index_to_key:
    if word!=query:                   # cosine similarity of a word to itself is 1; it will always be the most similar to itself
      similarity_scores[word]=cosine_similarity(model[word], model[query])
  sorted_score = sorted(similarity_scores.items(), key=operator.itemgetter(1), reverse=True)
  for idx, (k, v) in enumerate(sorted_score[:k]):
            print("%s\t%s\t%.5f" % (idx,k,v))

In [ ]:
find_k_nearest_neighbors(glove_100, 'burger', 5)

In [ ]:
find_k_nearest_neighbors(glove_100, 'pen', 5)

You can try different values of k and different queries to explore the learned representation in the provided pre-trained model.

We want you to get familiar with word embeddings and how you can manipluate the vectors in the models.


---
**Deliverable 1**


Which term (by cosine similarity over glove vectors) is most similar to "burger"?

<ol type = A>

<li>pencil</li>
<li>car</li>
<li>movie</li>

</ol>

Fill in your answer in the code block below; your answer should be either "A", "B" or "C"


In [ ]:
Q0 = ""

### Q1: Implementation of SemAxis

As mentioned in class, [SemAxis](https://arxiv.org/pdf/1806.05521.pdf) is a method for scoring terms along a user-defined axis.  In this queston, you will be exploring vector semantics (as explained in [6.2](https://web.stanford.edu/~jurafsky/slp3/6.pdf)) and implementing this method.

Given a set of word embeddings for positive terms $S^+ = \{v_1^+, \ldots v_n^+\}$ and embeddings for negative terms $S^- = \{v_1^-, \ldots v_n^-\}$ that define the endpoints of the axis, a semantic axis is given as:

$$
\mathbf{V}^+ = {1 \over n} \sum_1^n v_i^+
$$

$$
\mathbf{V}^- = {1 \over m} \sum_1^m v_i^-
$$

$$
\mathbf{V}_{\textrm{axis}} = \mathbf{V}^+ - \mathbf{V}^-
$$

#### A: Implement `get_semaxis`





**Deliverable 2**

Using the information above, implement the function `get_semaxis`, which will take in a set of glove embeddings (`vectors`, a `KeyedVectors` object) as well as two lists of the positive and negative terms which define the endpoints of the axis. The function should return $\mathbf{V}_{\textrm{axis}}$, in the form of an `numpy.array`.

In [ ]:
def get_semaxis(vectors, positive_terms, negative_terms):
    '''
    vectors: KeyedVectors word embeddings
    positive_terms: list of terms (strings) defining one end of an axis
    negative_terms: list of terms (strings) defining the other end of an axis

    output: SemAxis vector (a numpy array)
    '''
    # TODO

    v_axis = ...

    return v_axis


#### B: Implement `get_semaxis_score`


**Deliverable 3**

Implement the function `get_semaxis_score` to calculate the score of a word along the specified semantic axis. See section 3.1.3 from the SemAxis paper for details, but this score is simply the cosine similarity between the term and the axis:

$$
score(w)_{\mathbf{V_\textrm{axis}}} = \textrm{cos}(v_w, \mathbf{V}_\textrm{axis})
$$

`get_semaxis_score` takes in a `semaxis` (the output of your `get_semaxis` function above) and a target word's vector. The function should return the score of where the target would land on the axis, in the form of a `float`.

In [ ]:
def get_semaxis_score(axis, target_vector):
    '''
    axis: SemAxis vector
    target_vector: target term's vector

    output: target vector's score on SemAxis (a float)
    '''

    # TODO

    return score

In this example, we are using "woman" and "women" as defining one end of an axis, and "man" and "men" as defining the other.

In [ ]:
# DO NOT CHANGE THIS CELL
axis=get_semaxis(glove_100, ["woman", "women"], ["man", "men"])
target_vector=glove_100['actress']
get_semaxis_score(axis, target_vector)



---



Now let's score a set of target terms along that axis. (*Just run the following, do not change.*)

In [ ]:
def score_list_of_targets(vectors, positive_terms=None, negative_terms=None, target_words=None):
    scores=[]

    axis=get_semaxis(vectors, positive_terms, negative_terms)

    for target in target_words:
        scores.append((get_semaxis_score(axis, glove_100[target]), target))

    for k,v in reversed(sorted(scores)):
        print("%.3f\t%s" % (k,v))

In [ ]:
targets=["doctor", "nurse", "actor", "actress", "mechanic", "librarian", "architect", "magician", "cook", "chef"]

In [ ]:
score_list_of_targets(glove_100, positive_terms=["woman", "women"], negative_terms=["man", "men"], target_words=targets)

Consider what these scores mean in context. What does having a larger score versus a lower one mean in context, given our SemAxis?  (Just reflect on this question yourself -- there is no deliverable for this reflection.)



---


### Q2: Debiasing

Implicit bias and prejudice that are present within the training data end up as biases within word embeddings, as described in [6.11 and 6.12](https://web.stanford.edu/~jurafsky/slp3/6.pdf).
One potential method of debiasing is presented within [Bolukbasi et al. 2016](https://arxiv.org/pdf/1607.06520). We will be exploring this method, as interpreted through [Vargas and Cotterell 2024](https://arxiv.org/abs/2009.09435).

In this formulation, debiasing involves the following 3 steps:

<ol type = A>

<li>creating "defining sets" which define the subspaces</li>
<li>get subspace axis through PCA</li>
<li>subtracting orthogonal projection onto that subspace from the original embeddings</li>

</ol>

#### A. Defining sets

In this exercise, let's define a gender subspace as our "defining sets" as what is below.

$$
D_1 = \{man, woman\}\\
D_2 = \{mr., mrs.\}
$$


In this example, following [Vargas and Cotterell 2024](https://arxiv.org/abs/2009.09435), we will be using these key terms in order to define and create a gender subspace through the creation of a new matrix $W$.


**Building and interpreting defining sets**


Using $e_{word}$ to denote the embedding for a word, this matrix should be made up of each defining term's embedding

> in $D_1$ our defining terms's embeddings would be $e_{man}, e_{woman}$

being subtracted by the average of the embeddings within the given set.


> $e_{man} - \textrm{mean}(e_{man},e_{woman})$ \
> $e_{woman} - \textrm{mean}(e_{man},e_{woman})$

This results in the following matrix given both the defining sets ($D_1, D_2$) above:

$$
W=
\begin{bmatrix}
e_{man} - \textrm{mean}(e_{man},e_{woman}) \\
e_{woman} - \textrm{mean}(e_{man},e_{woman})\\
e_{mr.} - \textrm{mean}(e_{mr.},e_{mrs.})\\
e_{mrs.} - \textrm{mean}(e_{mr.},e_{mrs.})\\
\end{bmatrix}
$$



---
**Deliverable 4**

Now, complete the `create_design_matrix` function below to construct this design matrix $W$ and return it. This function should output a single numpy.array of size $4 \times 100$, given our embeddings have 100 dimensions (as noted in Q0)

In [ ]:
def create_design_matrix(vectors, D1, D2):
    '''
    vectors: KeyedVectors word embeddings
    D1: list of terms (strings) comprises the first defining set
    D2: list of terms (strings) comprises the second defining set

    output: a numpy.array

    '''


In [ ]:
design_matrix=create_design_matrix(glove_100, ["man", "woman"], ["mr.", "mrs."])

#### B. PCA

Our next step is to find the bias subspace that exists within this data -- i.e., the vector that corresponds to the e.g. "gender" information encoded within it. As [Vargas and Cotterell 2024](https://arxiv.org/abs/2009.09435) note, the bias subspace is the first principle component of that matrix $W$, found by running [PCA](https://en.wikipedia.org/wiki/Principal_component_analysis) on it.

**Example**

As a refresher, here's how you run [PCA](https://en.wikipedia.org/wiki/Principal_component_analysis) on a random matrix to get the *first principle component*.

In [ ]:
fake_matrix=np.random.rand(3,3)
print("Fake matrix:")
print(fake_matrix)

Now that we have a random matrix, we will run PCA on it in order to extract the first principle component.

In [ ]:
pca=PCA(n_components=1).fit(fake_matrix)        # Set n_components to number of principle components desired (1)
pca_subspace=pca.components_[0]                 # Extract first principle component

print("First principle component:")
print(pca_subspace)

In [ ]:
# You'll see that this subspace is already normalized to unit length:
print(pca_subspace)
print(pca_subspace/np.sqrt(np.dot(pca_subspace, pca_subspace)))

Now, run [PCA](https://en.wikipedia.org/wiki/Principal_component_analysis) over the `design_matrix`, because the *gender subspace is the first principle component* of that process.


In [ ]:
pca=PCA(n_components=1).fit(design_matrix)
subspace=pca.components_[0]

#### C. Debias

To debias a vector with respect to a bias subspace, we need to subtract the information in the bias subspace from it. We do so by first finding the orthogonal projection of the original vector onto that subspace.

We find the orthogoal projection of any unit-normalized vector $w$ onto a subspace $b$ by:

$$
w_b = \textrm{dot}(w,b) \; b
$$

If $b$ and $x$ are 100 dimensions, $w_b$ is 100 dimensions too.

Debiasing a vector is simply removing the information about that subspace from it. The debiased vector $w_d$ is then simply $w - w_b$.  


---
**Deliverable 5**

Implement the `debias` function below, which should debias any input vector `vec` with respect to the input subspace `subspace`.

*Note: glove embeddings are not normalized ahead of time, so be sure to normalize them before carrying out your projection. Vector $v$ may be normalized by the following:*

$$v  \div \sqrt{\textrm{dot}(v,v)}$$

In [ ]:
def debias(vec, subspace):
    '''
    vec: vector to debias (numpy.array)
    subspace: bias subspace (the first principle component of PCA on the design matrix $W$) (numpy.array)

    output: a debiased vector (numpy.array )

    '''
    vec_norm=...     # TODO

    def project_onto_subspace(v, subspace):
        # TODO
        return ...

    projected_vec=project_onto_subspace(vec_norm, subspace)

    # TODO
    debiased=...

    return debiased

Debias the vectors for the targets used above and see if debiasing changes the association between these terms and gender semaxis used above. *(You can simply run the code from this point forward.)*

As a reminder, let's see the original scores and  ranking of our targets.

In [ ]:
score_list_of_targets(glove_100, positive_terms=["woman", "women"], negative_terms=["man", "men"], target_words=targets)

Now, let's see the new rankings of the same targets, after they were debiased.

In [ ]:
diffs={}
for term in targets:

    m,w=glove_100.cosine_similarities(debias(glove_100[term], subspace), [debias(glove_100["man"], subspace), debias(glove_100["woman"], subspace)])
    diffs[term]=w-m

for k, v in sorted(diffs.items(), key=lambda item: item[1], reverse=True):
    print("%.3f\t%s" % (v,k))

Take note of the differences in these scores, and what they may mean in terms of the influence of a biased dataset.

### Closing and Submission

Congratulations on finishing HW1! Please ensure that you submit the completed notebook (`.ipynb`) onto Gradescope before February 4 at 11:59pm.  The notebook you upload to Gradescope must be named **HW1.ipynb**.

`File` --> `Download` --> `Download .ipynb`